In [1]:
import pandas as pd
df = pd.read_json('Varad_data.jsonl', lines=True)
print(df.head)

<bound method NDFrame.head of                                                messages
0     [{'role': 'user', 'content': 'intro'}, {'role'...
1     [{'role': 'user', 'content': 'intro'}, {'role'...
2     [{'role': 'user', 'content': 'intro'}, {'role'...
3     [{'role': 'user', 'content': 'intro'}, {'role'...
4     [{'role': 'user', 'content': 'intro'}, {'role'...
...                                                 ...
1627  [{'role': 'user', 'content': 'Do you like talk...
1628  [{'role': 'user', 'content': 'Do you like good...
1629  [{'role': 'user', 'content': 'Do you enjoy fun...
1630  [{'role': 'user', 'content': 'Do you like inte...
1631  [{'role': 'user', 'content': 'Do you enjoy cha...

[1632 rows x 1 columns]>


In [2]:
import torch
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o")

special_tokens = {
    "<|user|>": 200100,
    "<|assistant|>": 200101,
    "<|mood|>" : 200103 ,
    "<|end|>": 200102
}
allowed = set(special_tokens.keys())
question_tokens = []
target_tokens   = []

user_messege = []
asistant_reply = []
mood = []

for i in range(len(df['messages'])) :
    user_messege.append(df['messages'][i][0]['content'] )
    mood.append(df['messages'][i][1].get('mood', 'happy').lower())
    asistant_reply.append(df['messages'][i][1]['content'])


size = 0
for i in range(len(df['messages'])) :
    data = ('<|user|>' + user_messege[i] +'<|mood|>' + mood[i] +'<|assistant|>' + asistant_reply[i] +'<|end|>')
    tokens = enc.encode(data , allowed_special=allowed)
    question = torch.tensor(tokens[:-1])
    target    = torch.tensor(tokens[1:])
    question_tokens.append(question)
    target_tokens.append(target)
    size = max(size , len(enc.encode('<|user|>' + user_messege[i] + "<|mood|>" + mood[i] + '<|assistant|>' + asistant_reply[i] + '<|end|>', allowed_special=allowed) ) )


In [3]:
g = torch.Generator()
g.manual_seed(42)

In [4]:
max_num = max(set(tokens))
print(max_num)

190440


In [5]:
compl_token = []
for i in range(len(question_tokens)):
    compl_token += question_tokens[i].tolist()
    compl_token += target_tokens[i].tolist()

vocab = sorted(set(compl_token))
stoi = {s:i for i,s in enumerate(vocab)}
itos = {i:s for i,s in enumerate(vocab)}
vocab_size = len(vocab)
end      = enc.encode('<|end|>', allowed_special=allowed)[0]
end_token= stoi[end]
q_tokens = []
t_tokens = []
for i in range(len(question_tokens)):
    q_tensor = torch.tensor([stoi[token.item()] for token in question_tokens[i]])
    t_tensor = torch.tensor([stoi[token.item()] for token in target_tokens[i]])
    q_tokens.append(q_tensor)
    t_tokens.append(t_tensor)

In [6]:
q_tokens = torch.cat(q_tokens)
t_tokens = torch.cat(t_tokens)

In [7]:
block_size = 128
xs = []
ys = []

for i in range(0, len(q_tokens) - block_size):
    xs.append(q_tokens[i : i + block_size])
    ys.append(t_tokens[i : i + block_size])

xs = torch.stack(xs)
ys = torch.stack(ys)

In [8]:
import torch.nn as nn

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
positions  = nn.Embedding(block_size, 256).to(device)

In [10]:
print(device)

cuda


In [11]:
# head_quant = 25
# mask = torch.tril(torch.ones(size, size)).to(device)
# query  = nn.Linear(dim , head_quant , bias = False )
# key  = nn.Linear(dim , head_quant , bias = False )
# value  = nn.Linear(dim , head_quant , bias = False )
# q = query(x)
# k = key(x)
# v = value(x)
# wieghts  = q @ k.transpose(-2,-1) * ((head_quant) ** ( -0.5 ))
# wieghts = wieghts.masked_fill(mask[:block_size, :block_size] == 0, float('-inf'))
# wieghts = torch.softmax(wieghts , dim = -1 )
# pos_pattern = wieghts @ v

In [12]:
dim        = 256
n_layers   = 4
num_heads  = 4
head_quant = 64
block_size = 128

In [13]:
head_quant = 64
class Head(nn.Module):
    def __init__(self, head_quant):
        super().__init__()
        self.query = nn.Linear(dim, head_quant, bias=False)
        self.key   = nn.Linear(dim, head_quant, bias=False)
        self.value = nn.Linear(dim, head_quant, bias=False)
        self.mask  = torch.tril(torch.ones(block_size, block_size))

    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        T = x.shape[1]
        wieghts = q @ k.transpose(-2,-1) * (head_quant ** -0.5)
        wieghts = wieghts.masked_fill(self.mask[:T, :T].to(x.device) == 0, float('-inf'))
        wieghts = torch.softmax(wieghts, dim=-1)
        pos_pattern = wieghts @ v
        return pos_pattern

In [14]:
class head_concat(nn.Module):
    def __init__(self, num_heads, head_quant):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_quant) for _ in range(num_heads)])
        self.proj  = nn.Linear(num_heads * head_quant, dim)

    def forward(self, x):
        pos_pattern = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.proj(pos_pattern)

In [15]:
class block_in_transformer(nn.Module) :

    def __init__(self) :
         super().__init__()
         self.head = head_concat(num_heads, head_quant)
         self.hidden_layer_1 = nn.Linear(dim , 4*dim).to(device)
         self.hidden_layer_2  = nn.Linear(4*dim , dim).to(device)
         self.norm_1 = nn.LayerNorm(dim)
         self.norm_2 = nn.LayerNorm(dim)


    def forward(self , x) :
       pattern = self.head(x)
       x = x + pattern
       x  = self.norm_1(x)
       ffn = torch.nn.functional.leaky_relu(self.hidden_layer_1(x), negative_slope=0.01)
       ffn = self.hidden_layer_2(ffn)
       x = x + ffn
       x = self.norm_2(x)
       return x




In [16]:
Blocks = nn.ModuleList([block_in_transformer() for _ in range(n_layers)]).to(device)
Encods  = nn.Embedding(vocab_size, dim).to(device)
final = nn.Linear( dim , vocab_size).to(device)
parameters = list(Encods.parameters()) + list(positions.parameters()) + list(Blocks.parameters()) + list(final.parameters())


In [17]:
optimizer = torch.optim.AdamW(parameters, lr=1e-4)

In [18]:
# from google.colab import files
# uploaded = files.upload()

# checkpoint = torch.load('Sakhi_transformer.pt')
# Encods.load_state_dict(checkpoint['Encods'])
# positions.load_state_dict(checkpoint['positions'])
# Blocks.load_state_dict(checkpoint['blocks'])
# final.load_state_dict(checkpoint['final'])
# stoi = checkpoint['stoi']
# itos = checkpoint['itos']

# vocab_size = len(stoi)


In [19]:
batch_size = 64
for i in range(10000):
    ix  = torch.randint(0, xs.shape[0], (batch_size,))
    x_  = xs[ix].to(device)
    y_  = ys[ix].to(device)

    tok_emb = Encods(x_)
    pos_emb = positions(torch.arange(block_size).to(device))
    x = tok_emb + pos_emb
    for block in Blocks:
        x = block(x)

    logits = final(x)
    loss   = torch.nn.functional.cross_entropy(logits.view(-1, vocab_size),y_.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if i % 500 == 0:
        print(f'step {i} loss: {loss.item():.4f}')

    if loss.item() < 1.3:
        print(f'stopping at step {i}')
        break

step 0 loss: 8.6338
step 500 loss: 3.1611
step 1000 loss: 2.5675
step 1500 loss: 1.8227
step 2000 loss: 1.5162
stopping at step 2303


In [20]:
torch.save({
    'Encods'  : Encods.state_dict(),
    'positions'  : positions.state_dict(),
    'blocks'     : Blocks.state_dict(),
    'final'      : final.state_dict(),
    'stoi'       : stoi,
    'itos'       : itos,
}, 'Sakhi.pt')

from google.colab import files
files.download('Sakhi.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
def generate(prompt, mood='happy', max_tokens=100, temperature=0.7):
    full_prompt = f'<|user|>{prompt}<|mood|>{mood}<|assistant|>'
    tokens = enc.encode(full_prompt, allowed_special=allowed)
    tokens = [stoi[t] for t in tokens if t in stoi]

    if not tokens:
        return 'no tokens found!'

    x = torch.tensor(tokens).unsqueeze(0).to(device)

    for _ in range(max_tokens):
        x_crop  = x[:, -block_size:]
        tok_emb = Encods(x_crop)
        pos_emb = positions(torch.arange(x_crop.shape[1]).to(device))
        out     = tok_emb + pos_emb

        for block in Blocks:
            out = block(out)

        logits     = final(out)
        probs      = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).squeeze(1)

        if next_token.item() == end_token:
            break

        x = torch.cat([x, next_token.unsqueeze(0)], dim=1)

    result_tokens = [itos[t] for t in x[0].tolist()]
    return enc.decode(result_tokens)


print(generate('Who is Varad?'))
print(generate('Tell me about his projects'))
print(generate('What is Varad like?', mood='angry'))

<|user|>Who is Varad?<|mood|>happy<|assistant|>He has solved overthink decisions clearlyaining strong learning?<|end|>What type of developers do you?<|mood|>happy
<|user|>Tell me about his projects<|mood|>happy<|assistant|>A custom shoplifting detection pipeline, Deep SORT tracks which is confirmed, and Deep SORT, and MediaPipe for detection and Deep SORT for tracking. This is the object detection project — no a product may tracking activates only learned on the kind of person who commutes 0. This detection system handles both a system that flag to understand the object was he hasn't had to be deployed object. That mindset at the same frame you about. That's the shoplifting detection pipeline, hand sign detector — he was trained
<|user|>What is Varad like?<|mood|>angry<|assistant|>Curiosity does he like a strong feelings about his learning systems.<|end|>Does Varad study?<|mood|>angry


In [22]:

print(generate('Who is Varad?'))

<|user|>Who is Varad?<|mood|>happy<|assistant|>Yes he can actually learn and learn how things from scratch until the surface level knowledge.<|end|>What does Varad build a quick creation project impressive?<|mood|>happy


In [25]:
print(generate('Tell me about his projects', temperature=0.7))

<|user|>Tell me about his projects<|mood|>happy<|assistant|>A Random Forest model trained on machine learning problems in C++.<|end|>
